In [173]:
import numpy as np
import pandas as pd
import re

In [174]:
df = pd.read_csv("data/election_results.csv")

In [175]:
df.tail(60).values

array([['占冠村長選挙', '2021/8/29', '北海道', '田中\u3000正治', nan, 65.0, '男',
        '無所属', '現', '占冠村長'],
       ['高浜市議会議員補欠選挙', '2021/8/29', '愛知県', '杉浦\u3000浩一', nan, 51.0, '男',
        '無所属', '新', '会社取締役'],
       ['宇和島市長選挙', '2021/8/29', '愛媛県', '岡原\u3000文彰', nan, 51.0, '男',
        '無所属', '現', '宇和島市長'],
       ['高浜市長選挙', '2021/8/29', '愛知県', '吉岡\u3000初浩', nan, 65.0, '男',
        '無所属', '現', '会社取締役、高浜市長'],
       ['安芸市議会議員補欠選挙', '2021/8/22', '高知県', '佐藤\u3000倫与', 2983.0, 41.0,
        '女', '無所属', '新', '行政書士'],
       ['根室市議会議員選挙', '2021/8/22', '北海道', '橋本\u3000竜一', 1005.0, 44.0, nan,
        '共産', '現', '政党役員'],
       ['根室市議会議員選挙', '2021/8/22', '北海道', '須崎\u3000和貴', 837.0, 26.0, nan,
        '無所属', '新', '小売従業員'],
       ['根室市議会議員選挙', '2021/8/22', '北海道', '壺田\u3000重夫', 833.0, 71.0, nan,
        '無所属', '現', '国際映画音楽株式会社代表取締役'],
       ['根室市議会議員選挙', '2021/8/22', '北海道', '田塚\u3000不二男', 802.0, 74.0, nan,
        '公明', '現', '根室市議会議員'],
       ['根室市議会議員選挙', '2021/8/22', '北海道', '中村\u3000久', 789.0, 56.0, nan

In [176]:
df.columns

Index(['選挙名', '投票日', '都道府県', '氏名', '得票数', '年齢', '性別', '党派', '新旧', '肩書'], dtype='object')

In [177]:
df.shape

(79575, 10)

In [178]:
def create_row(group):
    
    name, date = group.name

    city_match = re.search(r'^(.+?[市区町村])', name)
    city_name = city_match.group(1) if city_match else "不明"

    if "議員" in name:
        election_type = "議員"
    elif re.search(r'([市区町村]長|知事)', name):
        election_type = "首長"
    else:
        election_type = "その他"

    is_special = 0
    pref_pattern = r'^(東京都|北海道|(京都|大阪)府|.{2,3}県)'
    if "補欠選挙" in name:
        is_special = 1
    elif re.search(pref_pattern + r'(?!.*[市区町村])', name) or "知事" in name:
        is_special = 1
        # 都道府県名を取得して市区町村変数に入れる
        pref_match = re.search(pref_pattern, name)
        if pref_match:
            city_name = pref_match.group(1)
    
    res = {}
    res['市区町村'] = city_name
    res['種別'] = election_type
    res['日付'] = date
    res['候補者数'] = group.shape[0]
    res['年齢中央値'] = group['年齢'].median()
    res['年齢最小値'] = group['年齢'].min()
    res['男性'] = group['性別'].eq('男').sum()
    res['女性'] = group['性別'].eq('女').sum()
    res['新人割合'] = group['新旧'].eq('新').sum() / group.shape[0] if group.shape[0] > 0 else 0
    res['65歳以上割合'] = group['年齢'].ge(65).sum() / group.shape[0] if group.shape[0] > 0 else 0
    res['自民+公明割合'] = group['党派'].str.contains('自民|公明').sum() / group.shape[0] if group.shape[0] > 0 else 0
    res['共産+社民割合'] = group['党派'].str.contains('共産|社民').sum() / group.shape[0] if group.shape[0] > 0 else 0
    res['無所属割合'] = group['党派'].eq('無所属').sum() / group.shape[0] if group.shape[0] > 0 else 0
    res['特別カテゴリ'] = is_special

    # Series で返すと、最終的に DataFrame にまとまります
    return pd.Series(res)

# グループ化して適用
new_df = df.groupby(['選挙名', '投票日']).apply(create_row).reset_index()

print(new_df)

c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\nshun\anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=ke

                選挙名         投票日   市区町村  種別          日付  候補者数  年齢中央値  年齢最小値  \
0     あきる野市議会議員補欠選挙   2015/10/4  あきる野市  議員   2015/10/4     2   49.5   44.0   
1       あきる野市議会議員選挙   2013/6/23  あきる野市  議員   2013/6/23    21   60.0   35.0   
2       あきる野市議会議員選挙   2017/6/11  あきる野市  議員   2017/6/11    21   52.0   39.0   
3          あきる野市長選挙   2015/10/4  あきる野市  首長   2015/10/4     1   66.0   66.0   
4          あきる野市長選挙   2019/10/6  あきる野市  首長   2019/10/6     1   62.0   62.0   
...             ...         ...    ...  ..         ...   ...    ...    ...   
9567      龍郷町議会議員選挙   2016/8/28    龍郷町  議員   2016/8/28    10   58.0   38.0   
9568      龍郷町議会議員選挙   2020/8/30    龍郷町  議員   2020/8/30    10   61.5   42.0   
9569         龍郷町長選挙  2013/10/20    龍郷町  首長  2013/10/20     1   53.0   53.0   
9570         龍郷町長選挙  2017/10/22    龍郷町  首長  2017/10/22     1   65.0   65.0   
9571         龍郷町長選挙  2021/10/31    龍郷町  首長  2021/10/31     1   69.0   69.0   

      男性  女性      新人割合   65歳以上割合   自民+公明割合   共産+社民割合     無所属割合 

C:\Users\nshun\AppData\Local\Temp\ipykernel_16832\2132680152.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df = df.groupby(['選挙名', '投票日']).apply(create_row).reset_index()


In [179]:
new_df.loc[new_df['市区町村'] == '不明', :]

,選挙名,投票日,市区町村,種別,日付,候補者数,年齢中央値,年齢最小値,男性,女性,新人割合,65歳以上割合,自民+公明割合,共産+社民割合,無所属割合,特別カテゴリ
1029,京都府議会議員補欠選挙,2014/4/6,不明,議員,2014/4/6,3,36.0,34.0,2,1,1.000000,0.000000,1.000000,0.000000,0.000000,1
1030,京都府議会議員補欠選挙,2018/4/8,不明,議員,2018/4/8,2,52.0,42.0,2,0,1.000000,0.000000,1.000000,0.000000,0.000000,1
3433,宮城県議会議員補欠選挙,2021/10/31,不明,議員,2021/10/31,2,63.0,58.0,2,0,0.500000,0.500000,1.000000,0.000000,0.000000,1
3527,富山県議会議員補欠選挙,2016/10/23,不明,議員,2016/10/23,3,61.0,58.0,3,0,1.000000,0.333333,1.000000,0.000000,0.000000,1
3827,山口県議会議員補欠選挙,2014/2/23,不明,議員,2014/2/23,3,40.0,33.0,3,0,1.000000,0.000000,1.000000,0.000000,0.000000,1
3828,山口県議会議員補欠選挙,2018/2/4,不明,議員,2018/2/4,2,35.5,33.0,2,0,1.000000,0.000000,1.000000,0.000000,0.000000,1
4304,広島県議会議員補欠選挙,2013/11/10,不明,議員,2013/11/10,2,36.5,29.0,1,1,1.000000,0.000000,1.000000,0.000000,0.000000,1
4751,新潟県議会議員補欠選挙,2018/6/10,不明,議員,2018/6/10,2,52.0,52.0,2,0,1.000000,0.000000,0.000000,0.000000,1.000000,1
5173,東京都議会議員補欠選挙,2012/12/16,不明,議員,2012/12/16,3,54.5,44.0,2,0,0.666667,0.333333,1.000000,0.000000,0.000000,1
5174,東京都議会議員補欠選挙,2016/7/31,不明,議員,2016/7/31,4,46.5,37.0,3,1,1.000000,0.000000,1.000000,0.000000,0.000000,1


In [180]:
new_df = new_df.drop(columns=['選挙名', '投票日'])
new_df = new_df.drop(new_df[new_df['特別カテゴリ'] == 1].index, inplace=False)
new_df = new_df.drop(new_df[new_df['市区町村'] == '不明'].index, inplace=False)

In [181]:
final_df = new_df.drop(columns=['特別カテゴリ'])

In [182]:
final_df.to_csv("data/election_data.csv", index=False)